# register-buffer — worked example 3: Module with Only Buffers and No Learnable Parameters

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `register-buffer`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `nn.Module` can have buffers without any `nn.Parameter` entries. This pattern is useful for lookup tables, precomputed position encodings, or fixed statistics that must travel with the model checkpoint. Such a module has an empty `.parameters()` but a populated `state_dict()` and `.buffers()`.

## Worked solution

**Step 1 — design the module.** We build a `SinusoidalEncoding` module that stores precomputed sine/cosine position encodings as a buffer. No learnable parameters are needed.

**Step 2 — precompute the encoding.** Use the standard formula: for dimension `d`, `PE[pos, 2i] = sin(pos / 10000^(2i/d_model))` and `PE[pos, 2i+1] = cos(...)`. Compute this as a tensor.

**Step 3 — register as buffer.** Call `self.register_buffer('pe', pe_tensor)`. Now `pe` moves with the model (e.g., to GPU) and is saved in checkpoints.

**Step 4 — forward.** Simply add `self.pe[:, :seq_len, :]` to the input.

**Step 5 — verify buffer/parameter counts.** `len(list(module.parameters()))` is 0. `len(list(module.buffers()))` is 1.

In [ ]:
import torch as t
import torch.nn as nn
import math

class SinusoidalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 64):
        super().__init__()
        pe = t.zeros(1, max_len, d_model)
        position = t.arange(0, max_len, dtype=t.float).unsqueeze(1)  # (max_len, 1)
        div_term = t.exp(t.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[0, :, 0::2] = t.sin(position * div_term)
        pe[0, :, 1::2] = t.cos(position * div_term[:d_model // 2])
        self.register_buffer('pe', pe)

    def forward(self, x: t.Tensor) -> t.Tensor:
        # x shape: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

# --- exercise and print ---
enc = SinusoidalEncoding(d_model=8, max_len=16)

print('Num parameters:', len(list(enc.parameters())))  # 0
print('Num buffers:   ', len(list(enc.buffers())))      # 1
print('state_dict keys:', list(enc.state_dict().keys()))  # ['pe']
print('pe shape:', enc.pe.shape)  # (1, 16, 8)
print('pe.requires_grad:', enc.pe.requires_grad)  # False

# Test forward
batch = t.zeros(2, 6, 8)
out = enc(batch)
print('output shape:', out.shape)  # (2, 6, 8)